In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Smart Retail Inventory and Sales Analytics") \
     .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving inventory.csv to inventory.csv
Saving products.csv to products.csv
Saving sales.csv to sales.csv
Saving stores.csv to stores.csv
Saving suppliers.json to suppliers.json


In [ ]:
stores_df = spark.read.option("header", True).option("inferSchema", True).csv("stores.csv")
stores_df.show()

+--------+--------------------+---------+-----------+-----------+------------+
|store_id|          store_name|     city|      state| store_type|manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya Reddy|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|Hypermarket|  Amit Kumar|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|Supermarket| Sneha Patel|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|Hypermarket|  Farhan Ali|
|    S106|     Metro Mart Pune|     Pune|Maharashtra| Mini Store|  Neha Singh|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala| Mini Store| Arjun Verma|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|Supermarket|  Meera Nair|
+--------+--------------------+---------+-----------+-----------+------------+



In [ ]:
products_df = spark.read.option("header", True).option("inferSchema", True).csv("products.csv")
products_df.show()

+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P103|  Television|Electronics|          LG|       S203|     45000|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|
|      P110|        Sofa|  Furniture|      Godrej|       S204|     32000|
|      P111|  Headphones|Electronics| 

In [ ]:
inventory_df = spark.read.option("header", True).option("inferSchema", True).csv("inventory.csv")
inventory_df.show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|
|       I1009|    S105|      P108|            50|           20| 2026-01-15|
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
|       I101

In [ ]:
sales_df = spark.read.option("header", True).option("inferSchema", True).csv("sales.csv")
sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010|    S101|      P104|2026-01-16|

In [ ]:
suppliers_df = spark.read.option("multiline", True).json("suppliers.json")
suppliers_df.show(truncate=False)

+---------+---------------------------------+------+-----------+------------------------+
|city     |contact                          |rating|supplier_id|supplier_name           |
+---------+---------------------------------+------+-----------+------------------------+
|Hyderabad|{techsource@mail.com, 9876500011}|4.5   |S201       |TechSource India        |
|Bangalore|{mobileworld@mail.com, NULL}     |4.2   |S202       |MobileWorld Distributors|
|Mumbai   |{NULL, 9876500013}               |4.4   |S203       |HomeTech Supply         |
|Delhi    |{urban@mail.com, 9876500014}     |4.0   |S204       |Urban Furniture Co      |
|Pune     |{NULL, NULL}                     |3.8   |S205       |Fashion Direct          |
+---------+---------------------------------+------+-----------+------------------------+



In [ ]:
stores_df.printSchema()
products_df.printSchema()
inventory_df.printSchema()
sales_df.printSchema()
suppliers_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: int

In [ ]:
print("Stores Count   :", stores_df.count())
print("Products Count :", products_df.count())
print("Inventory Count:", inventory_df.count())
print("Sales Count    :", sales_df.count())
print("Suppliers Count:", suppliers_df.count())

Stores Count   : 8
Products Count : 12
Inventory Count: 12
Sales Count    : 15
Suppliers Count: 5


In [ ]:
stores_df.write.mode("overwrite").parquet("bronze/stores")
products_df.write.mode("overwrite").parquet("bronze/products")
inventory_df.write.mode("overwrite").parquet("bronze/inventory")
sales_df.write.mode("overwrite").parquet("bronze/sales")
suppliers_df.write.mode("overwrite").parquet("bronze/suppliers")
print("Bronze layer saved.")
spark.read.parquet("bronze/stores").show()
spark.read.parquet("bronze/products").show()
spark.read.parquet("bronze/inventory").show()
spark.read.parquet("bronze/sales").show()
spark.read.parquet("bronze/suppliers").show(truncate=False)

Bronze layer saved.
+--------+--------------------+---------+-----------+-----------+------------+
|store_id|          store_name|     city|      state| store_type|manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya Reddy|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|Hypermarket|  Amit Kumar|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|Supermarket| Sneha Patel|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|Hypermarket|  Farhan Ali|
|    S106|     Metro Mart Pune|     Pune|Maharashtra| Mini Store|  Neha Singh|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala| Mini Store| Arjun Verma|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|Supermarket|  Meera Nair|
+--------+--------------------+---------+-----------+-----------+------------+

+----------+------------+------

In [ ]:
print("Products with missing supplier_id:")
products_df.filter(col("supplier_id").isNull()).show()

Products with missing supplier_id:
+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [ ]:
print("Inventory with missing stock_quantity:")
inventory_df.filter(col("stock_quantity").isNull()).show()

Inventory with missing stock_quantity:
+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [ ]:
print("Sales with missing sale_amount:")
sales_df.filter(col("sale_amount").isNull()).show()

Sales with missing sale_amount:
+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|         UPI|
+-------+--------+----------+----------+-------------+-----------+------------+



In [ ]:
print("Sales with missing payment_mode:")
sales_df.filter(col("payment_mode").isNull()).show()

Sales with missing payment_mode:
+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|        NULL|
+-------+--------+----------+----------+-------------+-----------+------------+



In [ ]:
inventory_df = inventory_df.fillna({"stock_quantity": 0})
print("Inventory after fill:")
inventory_df.show()

Inventory after fill:
+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|
|       I1009|    S105|      P108|            50|           20| 2026-01-15|
|       I1010|    S106|      P109|             0|            6| 20

In [ ]:
sales_df = sales_df.fillna({"sale_amount": 0})
print("Sales after fill (sale_amount):")
sales_df.show()

Sales after fill (sale_amount):
+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA1010

In [ ]:
sales_df = sales_df.fillna({"payment_mode": "Not Provided"})
print("Sales after fill (payment_mode):")
sales_df.show()

Sales after fill (payment_mode):
+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|         UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|
| SA101

In [ ]:
products_df = products_df.fillna({"supplier_id": "UNKNOWN"})
print("Products after fill (supplier_id):")
products_df.show()

Products after fill (supplier_id):
+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|
|      P103|  Television|Electronics|          LG|       S203|     45000|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|   Wildcraft|       S206|      2500|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|
|      P110|        Sofa|  Furniture|      Godrej|       S204|     32000|
|  

In [ ]:
products_df = products_df.withColumn(
    "data_quality_status",
      when(col("supplier_id") == "UNKNOWN", "Incomplete").otherwise("Valid")
)

inventory_df = inventory_df.withColumn(
        "data_quality_status",
         when(col("stock_quantity") == 0, "Incomplete").otherwise("Valid"))

sales_df = sales_df.withColumn(
          "data_quality_status",
            when((col("sale_amount") == 0) | (col("payment_mode") == "Not Provided"),"Incomplete").otherwise("Valid"))

products_df.show()
inventory_df.show()
sales_df.show()


+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P103|  Television|Electronics|          LG|       S203|     45000|              Valid|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|              Valid|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|              Valid|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|              Valid|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|              Valid|
|      P108|    Backpack|    Fashion|   Wildcraft|       S20

In [ ]:
products_df.write.mode("overwrite").parquet("silver/products")
inventory_df.write.mode("overwrite").parquet("silver/inventory")
sales_df.write.mode("overwrite").parquet("silver/sales")
print("Silver layer saved.")

spark.read.parquet("silver/products").show()
spark.read.parquet("silver/inventory").show()
spark.read.parquet("silver/sales").show()

Silver layer saved.
+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P103|  Television|Electronics|          LG|       S203|     45000|              Valid|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|              Valid|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|              Valid|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|              Valid|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|              Valid|
|      P108|    Backpack|    Fashion|   

In [ ]:
suppliers_flat_df = suppliers_df.select(
    "supplier_id",
    "supplier_name",
     "city",
      "rating",
       col("contact.phone").alias("phone"),
       col("contact.email").alias("email"))
suppliers_flat_df.show(truncate=False)

+-----------+------------------------+---------+------+----------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone     |email               |
+-----------+------------------------+---------+------+----------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011|techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |NULL      |mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013|NULL                |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014|urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |NULL      |NULL                |
+-----------+------------------------+---------+------+----------+--------------------+



In [ ]:
suppliers_flat_df = suppliers_flat_df.fillna({"phone": "Not Provided"})
print("After filling phone:")
suppliers_flat_df.show(truncate=False)

After filling phone:
+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013  |NULL                |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Not Provided|NULL                |
+-----------+------------------------+---------+------+------------+--------------------+



In [ ]:
suppliers_flat_df = suppliers_flat_df.fillna({"email": "Not Provided"})
print("After filling email:")
suppliers_flat_df.show(truncate=False)

After filling email:
+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013  |Not Provided        |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Not Provided|Not Provided        |
+-----------+------------------------+---------+------+------------+--------------------+



In [ ]:
suppliers_flat_df.write.mode("overwrite").parquet("silver/suppliers_flat")
print("Flattened suppliers saved.")
spark.read.parquet("silver/suppliers_flat").show(truncate=False)

Flattened suppliers saved.
+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013  |Not Provided        |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Not Provided|Not Provided        |
+-----------+------------------------+---------+------+------------+--------------------+



In [ ]:
product_supplier_df = products_df.join(suppliers_flat_df, on="supplier_id", how="left")
product_supplier_df.show(truncate=False)

+-----------+----------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|supplier_id|product_id|product_name|category   |brand       |unit_price|data_quality_status|supplier_name           |city     |rating|phone       |email               |
+-----------+----------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|S201       |P101      |Laptop      |Electronics|Lenovo      |65000     |Valid              |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |P102      |Mobile      |Electronics|Samsung     |25000     |Valid              |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |P103      |Television  |Electronics|LG          |45000     |Valid              |HomeTech Supply         |Mumbai   |4.4   |9876500013  |No

In [ ]:
inventory_product_df = inventory_df.join(
    products_df,
        on="product_id",
        how="left")
inventory_product_df.show(truncate=False)

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|category   |brand       |supplier_id|unit_price|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|P101      |I1001       |S101    |10            |5            |2026-01-10 |Valid              |Laptop      |Electronics|Lenovo      |S201       |65000     |Valid              |
|P102      |I1002       |S101    |25            |10           |2026-01-10 |Valid              |Mobile      |Electronics|Samsung     |S202       |25000     |Valid              |
|P104      |I1003       |S101    |3             |5            |2026-01-11 |Valid              |Office Chair|Furnitu

In [ ]:
sales_store_df = sales_df.join(
      stores_df,
      on="store_id",
      how="left"
)

sales_store_df.show(truncate=False)


+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|data_quality_status|store_name          |city     |state      |store_type |manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|S101    |SA1001 |P101      |2026-01-10|1            |65000      |UPI         |Valid              |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S101    |SA1002 |P102      |2026-01-10|2            |50000      |Card        |Valid              |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S102    |SA1003 |P101      |2026-01-11|1            |65000      |UPI         |Valid              |Metro Mart Bangalore|Bangalore|Karnataka  |Supermarket|Priya 

In [ ]:
sales_product_df = sales_df.join(
      products_df,
          on="product_id",
              how="left"
              )
sales_product_df.show(truncate=False)


+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|sale_id|store_id|sale_date |quantity_sold|sale_amount|payment_mode|data_quality_status|product_name|category   |brand       |supplier_id|unit_price|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|P101      |SA1001 |S101    |2026-01-10|1            |65000      |UPI         |Valid              |Laptop      |Electronics|Lenovo      |S201       |65000     |Valid              |
|P102      |SA1002 |S101    |2026-01-10|2            |50000      |Card        |Valid              |Mobile      |Electronics|Samsung     |S202       |25000     |Valid              |
|P101      |SA1003 |S102    |2026-01-11|1            |65000      |UPI         |Valid           

In [ ]:
retail_sales_df = sales_df \
    .join(stores_df, "store_id", "left") \
    .join(products_df, "product_id", "left")
retail_sales_df.show(truncate=False)

+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|store_id|sale_id|sale_date |quantity_sold|sale_amount|payment_mode|data_quality_status|store_name          |city     |state      |store_type |manager_name|product_name|category   |brand       |supplier_id|unit_price|data_quality_status|
+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|P101      |S101    |SA1001 |2026-01-10|1            |65000      |UPI         |Valid              |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|Laptop      |Electronics|Lenovo      |S201       |65000     |Valid              |
|P10

In [ ]:
invalid_supplier_df = products_df.join(
    suppliers_flat_df,
     on="supplier_id",
    how="left_anti")
invalid_supplier_df.show()

+-----------+----------+------------+-----------+---------+----------+-------------------+
|supplier_id|product_id|product_name|   category|    brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+---------+----------+-------------------+
|       S206|      P107|       Watch|    Fashion| Fastrack|      8000|              Valid|
|       S206|      P108|    Backpack|    Fashion|Wildcraft|      2500|              Valid|
|       S999|      P111|  Headphones|Electronics|     Sony|      3000|              Valid|
|    UNKNOWN|      P112|     T-Shirt|    Fashion|     Puma|      1500|         Incomplete|
+-----------+----------+------------+-----------+---------+----------+-------------------+



In [ ]:
invalid_inventory_product_df = inventory_df.join(
    products_df,
    on="product_id",
    how="left_anti")
invalid_inventory_product_df.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|              Valid|
+----------+------------+--------+--------------+-------------+-----------+-------------------+



In [ ]:
invalid_sales_product_df = sales_df.join(
      products_df,
      on="product_id",
      how="left_anti")
invalid_sales_product_df.show()


+----------+-------+--------+----------+-------------+-----------+------------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|        Cash|              Valid|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+



In [ ]:
invalid_sales_store_df = sales_df.join(
      stores_df,
          on="store_id",
          how="left_anti")
invalid_sales_store_df.show()


+--------+-------+----------+---------+-------------+-----------+------------+-------------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+



In [ ]:
from pyspark.sql.functions import *

In [ ]:
inventory_product_df = inventory_product_df.withColumn(
      "stock_status",
          when(
                  col("stock_quantity") <= col("reorder_level"),
                  "Reorder Required").otherwise("Sufficient Stock")
                              )
inventory_product_df.select(
    "inventory_id",
    "product_id",
    "stock_quantity",
    "reorder_level",
    "stock_status").show()

+------------+----------+--------------+-------------+----------------+
|inventory_id|product_id|stock_quantity|reorder_level|    stock_status|
+------------+----------+--------------+-------------+----------------+
|       I1001|      P101|            10|            5|Sufficient Stock|
|       I1002|      P102|            25|           10|Sufficient Stock|
|       I1003|      P104|             3|            5|Reorder Required|
|       I1004|      P101|             8|            5|Sufficient Stock|
|       I1005|      P103|             5|            4|Sufficient Stock|
|       I1006|      P105|             2|            5|Reorder Required|
|       I1007|      P106|            30|           10|Sufficient Stock|
|       I1008|      P107|             4|            5|Reorder Required|
|       I1009|      P108|            50|           20|Sufficient Stock|
|       I1010|      P109|             0|            6|Reorder Required|
|       I1011|      P110|             1|            3|Reorder Re

In [ ]:
products_df = products_df.withColumn(
      "price_category",
          when(col("unit_price") >= 50000, "Premium")
          .when(col("unit_price") >= 10000, "Standard")
          .otherwise("Budget"))
products_df.select(
          "product_id",
          "product_name",
          "unit_price",
           "price_category").show()

+----------+------------+----------+--------------+
|product_id|product_name|unit_price|price_category|
+----------+------------+----------+--------------+
|      P101|      Laptop|     65000|       Premium|
|      P102|      Mobile|     25000|      Standard|
|      P103|  Television|     45000|      Standard|
|      P104|Office Chair|      7000|        Budget|
|      P105| Study Table|     12000|      Standard|
|      P106|       Shoes|      4500|        Budget|
|      P107|       Watch|      8000|        Budget|
|      P108|    Backpack|      2500|        Budget|
|      P109|Refrigerator|     38000|      Standard|
|      P110|        Sofa|     32000|      Standard|
|      P111|  Headphones|      3000|        Budget|
|      P112|     T-Shirt|      1500|        Budget|
+----------+------------+----------+--------------+



In [ ]:
sales_df = sales_df.withColumn(
      "revenue_category",
      when(col("sale_amount") >= 50000, "High Revenue")
       .when(col("sale_amount") >= 15000, "Medium Revenue")
        .otherwise("Low Revenue"))
sales_df.select(
             "sale_id",
             "sale_amount",
              "revenue_category").show()


+-------+-----------+----------------+
|sale_id|sale_amount|revenue_category|
+-------+-----------+----------------+
| SA1001|      65000|    High Revenue|
| SA1002|      50000|    High Revenue|
| SA1003|      65000|    High Revenue|
| SA1004|      18000|  Medium Revenue|
| SA1005|       8000|     Low Revenue|
| SA1006|      12500|     Low Revenue|
| SA1007|      38000|  Medium Revenue|
| SA1008|      32000|  Medium Revenue|
| SA1009|      10000|     Low Revenue|
| SA1010|      14000|     Low Revenue|
| SA1011|          0|     Low Revenue|
| SA1012|      12000|     Low Revenue|
| SA1013|      16000|  Medium Revenue|
| SA1014|       7500|     Low Revenue|
| SA1015|      25000|  Medium Revenue|
+-------+-----------+----------------+



In [ ]:
sales_df = sales_df.withColumn(
      "month",
      month(col("sale_date")))
sales_df.select(
              "sale_id",
               "sale_date",
                "month").show()


+-------+----------+-----+
|sale_id| sale_date|month|
+-------+----------+-----+
| SA1001|2026-01-10|    1|
| SA1002|2026-01-10|    1|
| SA1003|2026-01-11|    1|
| SA1004|2026-01-12|    1|
| SA1005|2026-01-12|    1|
| SA1006|2026-01-13|    1|
| SA1007|2026-01-14|    1|
| SA1008|2026-01-15|    1|
| SA1009|2026-01-15|    1|
| SA1010|2026-01-16|    1|
| SA1011|2026-01-17|    1|
| SA1012|2026-01-18|    1|
| SA1013|2026-02-01|    2|
| SA1014|2026-02-02|    2|
| SA1015|2026-02-03|    2|
+-------+----------+-----+



In [ ]:
sales_df = sales_df.withColumn(
      "year",
      year(col("sale_date")))
sales_df.select(
              "sale_id",
              "sale_date",
              "month",
              "year").show()

+-------+----------+-----+----+
|sale_id| sale_date|month|year|
+-------+----------+-----+----+
| SA1001|2026-01-10|    1|2026|
| SA1002|2026-01-10|    1|2026|
| SA1003|2026-01-11|    1|2026|
| SA1004|2026-01-12|    1|2026|
| SA1005|2026-01-12|    1|2026|
| SA1006|2026-01-13|    1|2026|
| SA1007|2026-01-14|    1|2026|
| SA1008|2026-01-15|    1|2026|
| SA1009|2026-01-15|    1|2026|
| SA1010|2026-01-16|    1|2026|
| SA1011|2026-01-17|    1|2026|
| SA1012|2026-01-18|    1|2026|
| SA1013|2026-02-01|    2|2026|
| SA1014|2026-02-02|    2|2026|
| SA1015|2026-02-03|    2|2026|
+-------+----------+-----+----+



In [ ]:
inventory_product_df = inventory_product_df.withColumn(
      "inventory_value",
        col("stock_quantity") * col("unit_price")
)

inventory_product_df.select(
              "inventory_id",
               "product_name",
                "stock_quantity",
                "unit_price",
                "inventory_value").show()

+------------+------------+--------------+----------+---------------+
|inventory_id|product_name|stock_quantity|unit_price|inventory_value|
+------------+------------+--------------+----------+---------------+
|       I1001|      Laptop|            10|     65000|         650000|
|       I1002|      Mobile|            25|     25000|         625000|
|       I1003|Office Chair|             3|      7000|          21000|
|       I1004|      Laptop|             8|     65000|         520000|
|       I1005|  Television|             5|     45000|         225000|
|       I1006| Study Table|             2|     12000|          24000|
|       I1007|       Shoes|            30|      4500|         135000|
|       I1008|       Watch|             4|      8000|          32000|
|       I1009|    Backpack|            50|      2500|         125000|
|       I1010|Refrigerator|             0|     38000|              0|
|       I1011|        Sofa|             1|     32000|          32000|
|       I1012|      

In [ ]:
suppliers_flat_df = suppliers_flat_df.withColumn(
      "supplier_quality",
        when(col("rating") >= 4.5, "Excellent")
        .when(col("rating") >= 4.0, "Good")
         .otherwise("Average"))
suppliers_flat_df.select(
           "supplier_id",
            "supplier_name",
             "rating",
             "supplier_quality").show()

+-----------+--------------------+------+----------------+
|supplier_id|       supplier_name|rating|supplier_quality|
+-----------+--------------------+------+----------------+
|       S201|    TechSource India|   4.5|       Excellent|
|       S202|MobileWorld Distr...|   4.2|            Good|
|       S203|     HomeTech Supply|   4.4|            Good|
|       S204|  Urban Furniture Co|   4.0|            Good|
|       S205|      Fashion Direct|   3.8|         Average|
+-----------+--------------------+------+----------------+



In [ ]:
print(inventory_product_df.columns)

['product_id', 'inventory_id', 'store_id', 'stock_quantity', 'reorder_level', 'last_update', 'data_quality_status', 'product_name', 'category', 'brand', 'supplier_id', 'unit_price', 'data_quality_status', 'stock_status', 'inventory_value']


In [ ]:
inventory_product_df = inventory_df.alias("i").join(
    products_df.alias("p"),
    on="product_id",
    how="left").select(
                "i.*",
                "p.product_name",
                 "p.category",
                 "p.brand",
                  "p.supplier_id",
                  "p.unit_price",
                  "p.price_category")
inventory_product_df.show()


+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+--------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|price_category|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+--------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|       Premium|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|      Standard|
|      P104|       I1003|    S101|             3|            5| 2026-01-11|              Valid|Office Chair|  Furniture| Featherlite|       

In [55]:
products_df.write.mode("overwrite").parquet("silver/products")
inventory_product_df.write.mode("overwrite").parquet("silver/inventory")
sales_df.write.mode("overwrite").parquet("silver/sales")
suppliers_flat_df.write.mode("overwrite").parquet("silver/suppliers")
products_df.show()
inventory_product_df.show()
sales_df.show()
suppliers_flat_df.show()

+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|       Premium|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|      Standard|
|      P103|  Television|Electronics|          LG|       S203|     45000|              Valid|      Standard|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|              Valid|        Budget|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|              Valid|      Standard|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|              Valid|        Budget|
|      P107|       

In [56]:
from pyspark.sql.functions import *

In [57]:
stores_df.groupBy("state") \
    .count() \
    .orderBy(desc("count")) \
    .show()

+-----------+-----+
|      state|count|
+-----------+-----+
|Maharashtra|    2|
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
+-----------+-----+



In [58]:
products_df.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
|  Furniture|    3|
+-----------+-----+



In [59]:
products_df.groupBy("brand") .count() .orderBy(desc("count")) .show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [60]:
from pyspark.sql.functions import col

inventory_product_df = inventory_product_df.withColumn(
    "inventory_value",
    col("stock_quantity") * col("unit_price"))
inventory_product_df.select(
            "inventory_id",
             "product_name",
              "stock_quantity",
              "unit_price",
              "inventory_value").show()

+------------+------------+--------------+----------+---------------+
|inventory_id|product_name|stock_quantity|unit_price|inventory_value|
+------------+------------+--------------+----------+---------------+
|       I1001|      Laptop|            10|     65000|         650000|
|       I1002|      Mobile|            25|     25000|         625000|
|       I1003|Office Chair|             3|      7000|          21000|
|       I1004|      Laptop|             8|     65000|         520000|
|       I1005|  Television|             5|     45000|         225000|
|       I1006| Study Table|             2|     12000|          24000|
|       I1007|       Shoes|            30|      4500|         135000|
|       I1008|       Watch|             4|      8000|          32000|
|       I1009|    Backpack|            50|      2500|         125000|
|       I1010|Refrigerator|             0|     38000|              0|
|       I1011|        Sofa|             1|     32000|          32000|
|       I1012|      

In [61]:
inventory_product_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = false)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)
 |-- data_quality_status: string (nullable = false)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- price_category: string (nullable = true)
 |-- inventory_value: integer (nullable = true)



In [62]:
inventory_product_df.groupBy("store_id") .agg(
            sum("inventory_value").alias("total_inventory_value")
                ) .orderBy(desc("total_inventory_value")) .show()

+--------+---------------------+
|store_id|total_inventory_value|
+--------+---------------------+
|    S101|              1296000|
|    S102|               745000|
|    S103|               159000|
|    S105|               125000|
|    S104|                32000|
|    S107|                32000|
|    S106|                    0|
|    S108|                 NULL|
+--------+---------------------+



In [63]:
inventory_product_df.groupBy("category") \
    .agg(
         sum("inventory_value").alias("total_inventory_value")).orderBy(desc("total_inventory_value")) \
                        .show()

+-----------+---------------------+
|   category|total_inventory_value|
+-----------+---------------------+
|Electronics|              2020000|
|    Fashion|               292000|
|  Furniture|                77000|
|       NULL|                 NULL|
+-----------+---------------------+



In [64]:
from pyspark.sql.functions import *

inventory_product_df = inventory_product_df.withColumn(
    "stock_status",
      when(
          col("stock_quantity") <= col("reorder_level"),
              "Reorder Required").otherwise("Sufficient Stock"))
inventory_product_df.select(
      "inventory_id",
      "product_name",
      "stock_quantity",
      "reorder_level",
      "stock_status").show()
inventory_product_df.filter(
        col("stock_status") == "Reorder Required").count()

+------------+------------+--------------+-------------+----------------+
|inventory_id|product_name|stock_quantity|reorder_level|    stock_status|
+------------+------------+--------------+-------------+----------------+
|       I1001|      Laptop|            10|            5|Sufficient Stock|
|       I1002|      Mobile|            25|           10|Sufficient Stock|
|       I1003|Office Chair|             3|            5|Reorder Required|
|       I1004|      Laptop|             8|            5|Sufficient Stock|
|       I1005|  Television|             5|            4|Sufficient Stock|
|       I1006| Study Table|             2|            5|Reorder Required|
|       I1007|       Shoes|            30|           10|Sufficient Stock|
|       I1008|       Watch|             4|            5|Reorder Required|
|       I1009|    Backpack|            50|           20|Sufficient Stock|
|       I1010|Refrigerator|             0|            6|Reorder Required|
|       I1011|        Sofa|           

5

In [65]:
sales_df.agg(
      sum("sale_amount").alias("total_revenue")
      ).show()


+-------------+
|total_revenue|
+-------------+
|       373000|
+-------------+



In [66]:
sales_store_df.groupBy(
    "store_id",
    "store_name").agg(
     sum("sale_amount").alias("total_revenue")).orderBy(
    desc("total_revenue")).show(truncate=False)

+--------+--------------------+-------------+
|store_id|store_name          |total_revenue|
+--------+--------------------+-------------+
|S101    |Metro Mart Hyderabad|154000       |
|S102    |Metro Mart Bangalore|65000        |
|S106    |Metro Mart Pune     |38000        |
|S107    |Metro Mart Kochi    |32000        |
|S103    |Metro Mart Mumbai   |30000        |
|S104    |Metro Mart Chennai  |24000        |
|S105    |Metro Mart Delhi    |20000        |
|S108    |Metro Mart Jaipur   |10000        |
+--------+--------------------+-------------+



In [67]:
sales_store_df.groupBy(
      "city"
      ).agg(
      sum("sale_amount").alias("total_revenue")).orderBy(
      desc("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|       154000|
|Bangalore|        65000|
|     Pune|        38000|
|    Kochi|        32000|
|   Mumbai|        30000|
|  Chennai|        24000|
|    Delhi|        20000|
|   Jaipur|        10000|
+---------+-------------+



In [68]:
sales_product_df.groupBy(
      "category"
      ).agg(
      sum("sale_amount").alias("total_revenue")).orderBy(
      desc("total_revenue")).show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       243000|
|    Fashion|        62000|
|  Furniture|        58000|
|       NULL|        10000|
+-----------+-------------+



In [69]:
sales_product_df.groupBy(
      "product_id",
       "product_name"
        ).agg(sum("sale_amount").alias("total_revenue")).orderBy(
        desc("total_revenue")).show(truncate=False)

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|P101      |Laptop      |130000       |
|P102      |Mobile      |75000        |
|P109      |Refrigerator|38000        |
|P110      |Sofa        |32000        |
|P107      |Watch       |24000        |
|P108      |Backpack    |20000        |
|P106      |Shoes       |18000        |
|P104      |Office Chair|14000        |
|P105      |Study Table |12000        |
|P120      |NULL        |10000        |
|P103      |Television  |0            |
+----------+------------+-------------+



In [70]:
sales_df.groupBy(
    "payment_mode").agg(
     sum("sale_amount").alias("total_revenue")).orderBy(
     desc("total_revenue")).show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|         UPI|       190500|
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
+------------+-------------+



In [71]:
sales_product_df.groupBy(
    "product_id",
     "product_name").agg(
      sum("sale_amount").alias("total_revenue")).orderBy(
      desc("total_revenue")).limit(1).show(truncate=False)

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|P101      |Laptop      |130000       |
+----------+------------+-------------+



In [72]:
sales_store_df.groupBy(
      "store_id",
      "store_name").agg(
        sum("sale_amount").alias("total_revenue")).orderBy(desc("total_revenue")).limit(1).show(truncate=False)

+--------+--------------------+-------------+
|store_id|store_name          |total_revenue|
+--------+--------------------+-------------+
|S101    |Metro Mart Hyderabad|154000       |
+--------+--------------------+-------------+



In [73]:
sales_product_df.groupBy(
    "category").agg(
    sum("sale_amount").alias("total_revenue")).orderBy(
    desc("total_revenue")).limit(1).show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       243000|
+-----------+-------------+



In [74]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [75]:
product_revenue_df = sales_product_df.groupBy(
      "product_id","product_name").agg(
        sum("sale_amount").alias("total_revenue"))
product_rank_df = product_revenue_df.withColumn(
                  "rank",
                    rank().over(Window.orderBy(desc("total_revenue"))))
product_rank_df.show()


+----------+------------+-------------+----+
|product_id|product_name|total_revenue|rank|
+----------+------------+-------------+----+
|      P101|      Laptop|       130000|   1|
|      P102|      Mobile|        75000|   2|
|      P109|Refrigerator|        38000|   3|
|      P110|        Sofa|        32000|   4|
|      P107|       Watch|        24000|   5|
|      P108|    Backpack|        20000|   6|
|      P106|       Shoes|        18000|   7|
|      P104|Office Chair|        14000|   8|
|      P105| Study Table|        12000|   9|
|      P120|        NULL|        10000|  10|
|      P103|  Television|            0|  11|
+----------+------------+-------------+----+



In [76]:
store_revenue_df = sales_store_df.groupBy(
      "store_id",
      "store_name").agg(
      sum("sale_amount").alias("total_revenue"))
store_rank_df = store_revenue_df.withColumn(
      "rank",
       rank().over(Window.orderBy(desc("total_revenue"))))
store_rank_df.show()


+--------+--------------------+-------------+----+
|store_id|          store_name|total_revenue|rank|
+--------+--------------------+-------------+----+
|    S101|Metro Mart Hyderabad|       154000|   1|
|    S102|Metro Mart Bangalore|        65000|   2|
|    S106|     Metro Mart Pune|        38000|   3|
|    S107|    Metro Mart Kochi|        32000|   4|
|    S103|   Metro Mart Mumbai|        30000|   5|
|    S104|  Metro Mart Chennai|        24000|   6|
|    S105|    Metro Mart Delhi|        20000|   7|
|    S108|   Metro Mart Jaipur|        10000|   8|
+--------+--------------------+-------------+----+



In [77]:
category_product_revenue_df = sales_product_df.groupBy(
    "category",
     "product_id",
      "product_name").agg(
      sum("sale_amount").alias("total_revenue"))
category_window = Window.partitionBy(
        "category").orderBy(
          desc("total_revenue"))
category_product_rank_df = category_product_revenue_df.withColumn(
          "rank",
            rank().over(category_window))
category_product_rank_df.show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|Electronics|      P102|      Mobile|        75000|   2|
|Electronics|      P109|Refrigerator|        38000|   3|
|Electronics|      P103|  Television|            0|   4|
|    Fashion|      P107|       Watch|        24000|   1|
|    Fashion|      P108|    Backpack|        20000|   2|
|    Fashion|      P106|       Shoes|        18000|   3|
|  Furniture|      P110|        Sofa|        32000|   1|
|  Furniture|      P104|Office Chair|        14000|   2|
|  Furniture|      P105| Study Table|        12000|   3|
+-----------+----------+------------+-------------+----+



In [78]:
category_product_rank_df.filter(
    col("rank") == 1
    ).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|    Fashion|      P107|       Watch|        24000|   1|
|  Furniture|      P110|        Sofa|        32000|   1|
+-----------+----------+------------+-------------+----+



In [79]:
category_product_rank_df.filter(
    col("rank") <= 3
    ).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|Electronics|      P102|      Mobile|        75000|   2|
|Electronics|      P109|Refrigerator|        38000|   3|
|    Fashion|      P107|       Watch|        24000|   1|
|    Fashion|      P108|    Backpack|        20000|   2|
|    Fashion|      P106|       Shoes|        18000|   3|
|  Furniture|      P110|        Sofa|        32000|   1|
|  Furniture|      P104|Office Chair|        14000|   2|
|  Furniture|      P105| Study Table|        12000|   3|
+-----------+----------+------------+-------------+----+



In [80]:
store_state_revenue_df = retail_sales_df.groupBy(
      "state",
       "store_id",
        "store_name").agg(
          sum("sale_amount").alias("total_revenue"))

In [81]:
state_window = Window.partitionBy(
      "state"
      ).orderBy(
      desc("total_revenue"))
top_store_state_df = store_state_revenue_df.withColumn(
      "rank",
      rank().over(state_window))
top_store_state_df.filter(
        col("rank") == 1).show(truncate=False)

+-----------+--------+--------------------+-------------+----+
|state      |store_id|store_name          |total_revenue|rank|
+-----------+--------+--------------------+-------------+----+
|Delhi      |S105    |Metro Mart Delhi    |20000        |1   |
|Karnataka  |S102    |Metro Mart Bangalore|65000        |1   |
|Kerala     |S107    |Metro Mart Kochi    |32000        |1   |
|Maharashtra|S106    |Metro Mart Pune     |38000        |1   |
|Rajasthan  |S108    |Metro Mart Jaipur   |10000        |1   |
|Tamil Nadu |S104    |Metro Mart Chennai  |24000        |1   |
|Telangana  |S101    |Metro Mart Hyderabad|154000       |1   |
+-----------+--------+--------------------+-------------+----+



In [82]:
daily_sales_df = sales_df.groupBy(
      "sale_date"
      ).agg(sum("sale_amount").alias("daily_revenue"))



In [83]:
running_window = Window.orderBy(
      "sale_date"
      )
running_total_df = daily_sales_df.withColumn(
          "running_total",
           sum("daily_revenue").over(
           running_window.rowsBetween(
           Window.unboundedPreceding,
          Window.currentRow)))
running_total_df.show()



+----------+-------------+-------------+
| sale_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2026-01-10|       115000|       115000|
|2026-01-11|        65000|       180000|
|2026-01-12|        26000|       206000|
|2026-01-13|        12500|       218500|
|2026-01-14|        38000|       256500|
|2026-01-15|        42000|       298500|
|2026-01-16|        14000|       312500|
|2026-01-17|            0|       312500|
|2026-01-18|        12000|       324500|
|2026-02-01|        16000|       340500|
|2026-02-02|         7500|       348000|
|2026-02-03|        25000|       373000|
+----------+-------------+-------------+



In [84]:
lag_df = daily_sales_df.withColumn(
    "previous_day_sales",
     lag("daily_revenue", 1).over(
      Window.orderBy("sale_date")))
lag_df.show()

+----------+-------------+------------------+
| sale_date|daily_revenue|previous_day_sales|
+----------+-------------+------------------+
|2026-01-10|       115000|              NULL|
|2026-01-11|        65000|            115000|
|2026-01-12|        26000|             65000|
|2026-01-13|        12500|             26000|
|2026-01-14|        38000|             12500|
|2026-01-15|        42000|             38000|
|2026-01-16|        14000|             42000|
|2026-01-17|            0|             14000|
|2026-01-18|        12000|                 0|
|2026-02-01|        16000|             12000|
|2026-02-02|         7500|             16000|
|2026-02-03|        25000|              7500|
+----------+-------------+------------------+



In [85]:
lead_df = sales_df.withColumn(
      "next_sale_amount",
       lead("sale_amount", 1).over(
        Window.orderBy("sale_date")))
lead_df.select(
        "sale_id",
        "sale_date",
        "sale_amount",
        "next_sale_amount").show()

+-------+----------+-----------+----------------+
|sale_id| sale_date|sale_amount|next_sale_amount|
+-------+----------+-----------+----------------+
| SA1001|2026-01-10|      65000|           50000|
| SA1002|2026-01-10|      50000|           65000|
| SA1003|2026-01-11|      65000|           18000|
| SA1004|2026-01-12|      18000|            8000|
| SA1005|2026-01-12|       8000|           12500|
| SA1006|2026-01-13|      12500|           38000|
| SA1007|2026-01-14|      38000|           32000|
| SA1008|2026-01-15|      32000|           10000|
| SA1009|2026-01-15|      10000|           14000|
| SA1010|2026-01-16|      14000|               0|
| SA1011|2026-01-17|          0|           12000|
| SA1012|2026-01-18|      12000|           16000|
| SA1013|2026-02-01|      16000|            7500|
| SA1014|2026-02-02|       7500|           25000|
| SA1015|2026-02-03|      25000|            NULL|
+-------+----------+-----------+----------------+



In [86]:
product_sales_df = sales_product_df.select(
    "product_id",
    "product_name",
    "sale_date",
    "sale_amount")

In [87]:
product_window = Window.partitionBy(
    "product_id"
    ).orderBy("sale_date")
increase_df = product_sales_df.withColumn(
            "previous_sale_amount",
             lag("sale_amount").over(product_window))

In [88]:
increase_df.filter(
    col("sale_amount") >
    col("previous_sale_amount")).show(truncate=False)

+----------+------------+----------+-----------+--------------------+
|product_id|product_name|sale_date |sale_amount|previous_sale_amount|
+----------+------------+----------+-----------+--------------------+
|P107      |Watch       |2026-02-01|16000      |8000                |
+----------+------------+----------+-----------+--------------------+



In [89]:
stores_df.createOrReplaceTempView("stores")
products_df.createOrReplaceTempView("products")
inventory_product_df.createOrReplaceTempView("inventory")
sales_df.createOrReplaceTempView("sales")
suppliers_flat_df.createOrReplaceTempView("suppliers")

In [90]:
stores_df.createOrReplaceTempView("stores")
products_df.createOrReplaceTempView("products")
inventory_product_df.createOrReplaceTempView("inventory")
sales_df.createOrReplaceTempView("sales")
suppliers_flat_df.createOrReplaceTempView("suppliers")

In [91]:
spark.sql("""
SELECT *
FROM sales
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     

In [92]:
spark.sql("""
SELECT category,
COUNT(*) AS total_products
FROM products
GROUP BY category
""").show()

+-----------+--------------+
|   category|total_products|
+-----------+--------------+
|    Fashion|             4|
|Electronics|             5|
|  Furniture|             3|
+-----------+--------------+



In [93]:
spark.sql("""
SELECT store_id,
SUM(sale_amount) AS total_revenue
FROM sales
GROUP BY store_id
ORDER BY total_revenue DESC
""").show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S101|       154000|
|    S102|        65000|
|    S106|        38000|
|    S107|        32000|
|    S103|        30000|
|    S104|        24000|
|    S105|        20000|
|    S108|        10000|
+--------+-------------+



In [94]:
spark.sql("""
SELECT st.city,
SUM(sa.sale_amount) AS total_revenue
FROM sales sa
JOIN stores st
ON sa.store_id = st.store_id
GROUP BY st.city
ORDER BY total_revenue DESC
""").show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|       154000|
|Bangalore|        65000|
|     Pune|        38000|
|    Kochi|        32000|
|   Mumbai|        30000|
|  Chennai|        24000|
|    Delhi|        20000|
|   Jaipur|        10000|
+---------+-------------+



In [95]:
spark.sql("""
SELECT inventory_id,
product_id,
stock_quantity,
reorder_level
FROM inventory
WHERE stock_quantity <= reorder_level
""").show()

+------------+----------+--------------+-------------+
|inventory_id|product_id|stock_quantity|reorder_level|
+------------+----------+--------------+-------------+
|       I1003|      P104|             3|            5|
|       I1006|      P105|             2|            5|
|       I1008|      P107|             4|            5|
|       I1010|      P109|             0|            6|
|       I1011|      P110|             1|            3|
+------------+----------+--------------+-------------+



In [96]:
spark.sql("""
SELECT s.*
FROM sales s
LEFT JOIN products p
ON s.product_id = p.product_id
WHERE p.product_id IS NULL
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|              Valid|     Low Revenue|    1|2026|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+



In [97]:
spark.sql("""
SELECT p.*
FROM products p
LEFT JOIN suppliers s
ON p.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""").show()

+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|    brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+
|      P107|       Watch|    Fashion| Fastrack|       S206|      8000|              Valid|        Budget|
|      P108|    Backpack|    Fashion|Wildcraft|       S206|      2500|              Valid|        Budget|
|      P111|  Headphones|Electronics|     Sony|       S999|      3000|              Valid|        Budget|
|      P112|     T-Shirt|    Fashion|     Puma|    UNKNOWN|      1500|         Incomplete|        Budget|
+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+



In [98]:
spark.sql("""
SELECT product_id,
SUM(sale_amount) revenue
FROM sales
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 5
""").show()

+----------+-------+
|product_id|revenue|
+----------+-------+
|      P101| 130000|
|      P102|  75000|
|      P109|  38000|
|      P110|  32000|
|      P107|  24000|
+----------+-------+



In [99]:
spark.sql("""
SELECT payment_mode,
SUM(sale_amount) total_revenue
FROM sales
GROUP BY payment_mode
ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|         UPI|       190500|
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
+------------+-------------+



In [100]:
retail_sales_df = retail_sales_df.select(
      *[c for c in retail_sales_df.columns if c != "data_quality_status"])
retail_sales_df = retail_sales_df.withColumn(
      "data_quality_status",
        lit("Valid"))
retail_sales_df.write \
        .mode("overwrite") \
        .parquet("gold/sales")

In [101]:
retail_sales_df = retail_sales_df \
    .withColumn("month", month("sale_date")) \
    .withColumn("year", year("sale_date"))

In [102]:
retail_sales_df.write \
    .mode("overwrite") \
    .partitionBy("year","month") \
    .parquet("gold/sales_partitioned")

In [103]:
%%writefile sales_march_2026.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_mode
SA1016,S101,P101,2026-03-01,1,65000,UPI
SA1017,S102,P103,2026-03-02,1,45000,Card
SA1018,S105,P108,2026-03-03,2,5000,Cash
SA1019,S104,P107,2026-03-04,1,8000,UPI
SA1020,S103,P105,2026-03-05,1,12000,Card

Writing sales_march_2026.csv


In [104]:
incremental_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("sales_march_2026.csv")
incremental_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
| SA1017|    S102|      P103|2026-03-02|            1|      45000|        Card|
| SA1018|    S105|      P108|2026-03-03|            2|       5000|        Cash|
| SA1019|    S104|      P107|2026-03-04|            1|       8000|         UPI|
| SA1020|    S103|      P105|2026-03-05|            1|      12000|        Card|
+-------+--------+----------+----------+-------------+-----------+------------+



In [105]:
incremental_df.write \
    .mode("append") \
    .parquet("silver/sales")

In [106]:
silver_sales_df = spark.read.parquet("silver/sales")

incremental_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("sales_march_2026.csv")
updated_sales_df = silver_sales_df.unionByName(
      incremental_df,allowMissingColumns=True)
updated_sales_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     

In [107]:
print(updated_sales_df.count())

25


In [108]:
updated_sales_df.groupBy(
      "product_id").agg(
       sum("sale_amount").alias("revenue")).orderBy(
        desc("revenue")).show()

+----------+-------+
|product_id|revenue|
+----------+-------+
|      P101| 260000|
|      P103|  90000|
|      P102|  75000|
|      P107|  40000|
|      P109|  38000|
|      P105|  36000|
|      P110|  32000|
|      P108|  30000|
|      P106|  18000|
|      P104|  14000|
|      P120|  10000|
+----------+-------+



In [109]:
updated_sales_df.groupBy(
    "store_id").agg(
     sum("sale_amount").alias("revenue")).orderBy(
     desc("revenue")).show()

+--------+-------+
|store_id|revenue|
+--------+-------+
|    S101| 284000|
|    S102| 155000|
|    S103|  54000|
|    S104|  40000|
|    S106|  38000|
|    S107|  32000|
|    S105|  30000|
|    S108|  10000|
+--------+-------+



In [110]:
updated_sales_df = updated_sales_df \
    .withColumn("month", month("sale_date")) \
    .withColumn("year", year("sale_date"))
updated_sales_df.write \
    .mode("overwrite") \
    .partitionBy("year","month") \
    .parquet("gold/sales_partitioned")

In [111]:
before_count = sales_df.count()
after_count = updated_sales_df.count()
print("Before Load :", before_count)
print("After Load  :", after_count)
print("New Records :", after_count - before_count)

Before Load : 15
After Load  : 25
New Records : 10


In [112]:
store_performance_df = retail_sales_df.groupBy(
    "store_id",
     "store_name",
      "city",
      "state").agg(
       count("sale_id").alias("total_sales"),
       sum("sale_amount").alias("total_revenue"))
store_performance_df.show(truncate=False)

+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|store_name          |city     |state      |total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|S106    |Metro Mart Pune     |Pune     |Maharashtra|1          |38000        |
|S107    |Metro Mart Kochi    |Kochi    |Kerala     |1          |32000        |
|S101    |Metro Mart Hyderabad|Hyderabad|Telangana  |4          |154000       |
|S103    |Metro Mart Mumbai   |Mumbai   |Maharashtra|2          |30000        |
|S105    |Metro Mart Delhi    |Delhi    |Delhi      |2          |20000        |
|S102    |Metro Mart Bangalore|Bangalore|Karnataka  |2          |65000        |
|S104    |Metro Mart Chennai  |Chennai  |Tamil Nadu |2          |24000        |
|S108    |Metro Mart Jaipur   |Jaipur   |Rajasthan  |1          |10000        |
+--------+--------------------+---------+-----------+-----------+-------------+



In [113]:
store_performance_df.write \
    .mode("overwrite") \
    .parquet("gold/store_performance")

In [114]:
product_performance_df = retail_sales_df.groupBy(
      "product_id",
      "product_name",
       "category",
        "brand").agg(
        sum("quantity_sold").alias("total_quantity_sold"),sum("sale_amount").alias("total_revenue"))
product_performance_df.show(truncate=False)

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|category   |brand       |total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|P110      |Sofa        |Furniture  |Godrej      |1                  |32000        |
|P104      |Office Chair|Furniture  |Featherlite |2                  |14000        |
|P101      |Laptop      |Electronics|Lenovo      |2                  |130000       |
|P109      |Refrigerator|Electronics|Whirlpool   |1                  |38000        |
|P105      |Study Table |Furniture  |Urban Ladder|1                  |12000        |
|P107      |Watch       |Fashion    |Fastrack    |3                  |24000        |
|P102      |Mobile      |Electronics|Samsung     |3                  |75000        |
|P108      |Backpack    |Fashion    |Wildcraft   |8                  |20000        |
|P106      |Shoes       |Fashion    |Nike        |4              

In [115]:
product_performance_df.write \
    .mode("overwrite") \
    .parquet("gold/product_performance")

In [116]:
inventory_reorder_df = inventory_product_df.select(
    "store_id",
    "product_id",
    "product_name",
    "stock_quantity",
    "reorder_level",
    "stock_status").filter(
            col("stock_status") == "Reorder Required")
inventory_reorder_df.show(truncate=False)

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|stock_status    |
+--------+----------+------------+--------------+-------------+----------------+
|S101    |P104      |Office Chair|3             |5            |Reorder Required|
|S103    |P105      |Study Table |2             |5            |Reorder Required|
|S104    |P107      |Watch       |4             |5            |Reorder Required|
|S106    |P109      |Refrigerator|0             |6            |Reorder Required|
|S107    |P110      |Sofa        |1             |3            |Reorder Required|
+--------+----------+------------+--------------+-------------+----------------+



In [117]:
inventory_reorder_df.write \
    .mode("overwrite") \
    .parquet("gold/inventory_reorder")

In [118]:
supplier_quality_report_df = suppliers_flat_df.select(
      "supplier_id",
      "supplier_name",
      "city",
      "rating",
      "supplier_quality",
      "phone",
      "email")
supplier_quality_report_df.show(truncate=False)

+-----------+------------------------+---------+------+----------------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|supplier_quality|phone       |email               |
+-----------+------------------------+---------+------+----------------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |Excellent       |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Good            |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |Good            |9876500013  |Not Provided        |
|S204       |Urban Furniture Co      |Delhi    |4.0   |Good            |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Average         |Not Provided|Not Provided        |
+-----------+------------------------+---------+------+----------------+------------+--------------------+



In [119]:
supplier_quality_report_df.write \
    .mode("overwrite") \
    .parquet("gold/supplier_quality")

In [120]:
from pyspark.sql.functions import countDistinct
category_revenue_df = retail_sales_df.groupBy(
    "category"
).agg(
countDistinct("product_id").alias("total_products"),
      sum("quantity_sold").alias("total_quantity_sold"),
      sum("sale_amount").alias("total_revenue"))
category_revenue_df.show(truncate=False)

+-----------+--------------+-------------------+-------------+
|category   |total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|Fashion    |3             |15                 |62000        |
|NULL       |1             |2                  |10000        |
|Electronics|4             |7                  |243000       |
|Furniture  |3             |4                  |58000        |
+-----------+--------------+-------------------+-------------+



In [121]:
category_revenue_df.write \
    .mode("overwrite") \
    .parquet("gold/category_revenue")

In [122]:
payment_mode_report_df = sales_df.groupBy(
      "payment_mode").agg(
       count("sale_id").alias("total_transactions"),
      sum("sale_amount").alias("total_revenue"))
payment_mode_report_df.show()

+------------+------------------+-------------+
|payment_mode|total_transactions|total_revenue|
+------------+------------------+-------------+
|        Card|                 5|       133000|
|        Cash|                 3|        35500|
|Not Provided|                 1|        14000|
|         UPI|                 6|       190500|
+------------+------------------+-------------+



In [123]:
payment_mode_report_df.write \
    .mode("overwrite") \
    .parquet("gold/payment_mode_report")

In [124]:
print("Store Performance")
store_performance_df.show()
print("Product Performance")
product_performance_df.show()
print("Inventory Reorder")
inventory_reorder_df.show()
print("Supplier Quality")
supplier_quality_report_df.show()
print("Category Revenue")
category_revenue_df.show()
print("Payment Mode")
payment_mode_report_df.show()

Store Performance
+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|          store_name|     city|      state|total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|          1|        38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|          1|        32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|          4|       154000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|          2|        30000|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|          2|        20000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|          2|        65000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|          2|        24000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|          1|        10000|
+--------+--------------------+---------+-----------+-----------+-------------+

Product Performance
+